In [2]:
import pandas as pd


ruta_archivo = 'dataset.csv' 
df_musica = pd.read_csv(r"C:\Users\lenovo\Documents\ASemestre 4\Analisis\PROYECTO\dataset.csv")

def obtener_datos_cancion_local(nombre_cancion, df):
    # Buscar la canción ignorando mayúsculas y minúsculas
    # Asumimos que la columna se llama 'track_name' (es el estándar en Kaggle)
    resultado = df[df['track_name'].str.contains(nombre_cancion, case=False, na=False)]
    
    if resultado.empty:
        return {"error": f"No se encontró la canción '{nombre_cancion}' en el dataset."}
    
    # Si encuentra varias versiones, tomamos la primera (índice 0)
    cancion = resultado.iloc[0]
    
    # Armamos el mismo diccionario que tenías antes, extrayendo de la fila del Excel
    # Usamos .get() por si tu CSV tiene nombres de columna ligeramente distintos
    datos = {
        'id': cancion.get('track_id', 'Sin ID'),
        'nombre': cancion.get('track_name', nombre_cancion),
        'artista': cancion.get('artists', 'Artista Desconocido'),
        'duracion_ms': cancion.get('duration_ms', 0),
        'popularidad': cancion.get('popularity', 0), # Variable a predecir
        'danceability': cancion.get('danceability', 0.0),
        'energy': cancion.get('energy', 0.0),
        'tempo_bpm': cancion.get('tempo', 0.0),
        'valence': cancion.get('valence', 0.0)
    }
    return datos



In [3]:
import requests
from bs4 import BeautifulSoup

def scrape_billboard_top_10():
    url = "https://www.billboard.com/charts/hot-100/"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    canciones = []
    # Las clases de Billboard cambian a menudo, esta es la estructura general actual
    items = soup.find_all('div', class_='o-chart-results-list-row-container')
    
    for item in items[:10]: # Solo el top 10 para probar
        titulo = item.find('h3', id='title-of-a-story').text.strip()
        artista = item.find('span', class_='c-label').text.strip()
        canciones.append({'titulo': titulo, 'artista': artista})
        
    return pd.DataFrame(canciones)

df_billboard = scrape_billboard_top_10()

In [4]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

def limpiar_datos(df):
    # 1. Eliminar duplicados
    df = df.drop_duplicates(subset=['id_cancion'])
    
    # 2. Manejo de valores nulos
    # Llenar nulos en variables numéricas con la mediana
    num_cols = ['danceability', 'energy', 'tempo_bpm']
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    
    # Eliminar filas si falta el nombre o el artista
    df = df.dropna(subset=['nombre', 'artista'])
    
    # 3. Normalización de texto (para cruzar datos de Spotify y Billboard)
    df['nombre'] = df['nombre'].str.lower().str.strip()
    df['artista'] = df['artista'].str.lower().str.strip()
    
    # 4. Transformación de variables
    # Convertir duración de milisegundos a minutos
    if 'duracion_ms' in df.columns:
        df['duracion_minutos'] = df['duracion_ms'] / 60000
        df = df.drop(columns=['duracion_ms'])
    
    # 5. Escalar variables numéricas para el modelo predictivo
    scaler = MinMaxScaler()
    variables_a_escalar = ['danceability', 'energy', 'tempo_bpm', 'valence', 'duracion_minutos']
    df[variables_a_escalar] = scaler.fit_transform(df[variables_a_escalar])
    
    return df

# Asumiendo que df_raw es tu DataFrame consolidado
#df_limpio = limpiar_datos(df_raw)

In [11]:
# ==========================================
# ZONA DE PRUEBAS
# ==========================================
print(" PRUEBA DATASET LOCAL ")

# Busca una canción que sepas que es famosa para ver si está en tu archivo
cancion_prueba = "Positions"
mis_datos = obtener_datos_cancion_local(cancion_prueba, df_musica)

if "error" in mis_datos:
    print(mis_datos["error"])
else:
    print(f"Éxito! Datos de {mis_datos['nombre']} por {mis_datos['artista']}:")
    print(f"   - Popularidad: {mis_datos['popularidad']}/100")
    print(f"   - Bailable: {mis_datos['danceability']}")
    print(f"   - Energía: {mis_datos['energy']}")
    print(f"   - Tempo (BPM): {mis_datos['tempo_bpm']}")

# 2. Probamos Billboard
print("\n📈 --- PRUEBA BILLBOARD --- 📈")
try:
    df_top10 = scrape_billboard_top_10()
    print("¡Éxito! Aquí está el Top 5 actual:")
    print(df_top10.head(5))
except Exception as e:
    print("Error leyendo Billboard:", e)

 PRUEBA DATASET LOCAL 
Éxito! Datos de positions por Ariana Grande:
   - Popularidad: 85/100
   - Bailable: 0.737
   - Energía: 0.802
   - Tempo (BPM): 144.015

📈 --- PRUEBA BILLBOARD --- 📈
¡Éxito! Aquí está el Top 5 actual:
           titulo artista
0  Choosin' Texas       1
1          Be Her       2
2       Drop Dead       3
3      Man I Need       4
4    I Just Might       5
